In [13]:
import pandas as pd
import glob
from datetime import date, timedelta
import numpy as np
from datetime import datetime
import pathlib
from pathlib import Path
from collections import OrderedDict
import polars as pl
import fastexcel
import os
import time
import warnings
warnings.filterwarnings("ignore", message="Could not determine dtype")

In [14]:
def convert_to_datetime(struct_time):
    """Convert struct_time to datetime object."""
    return datetime(*struct_time[:6])

def input_data(folder_path, sheet_name=None):
    file_paths = glob.glob(f"{folder_path}/*.xlsx") + glob.glob(f"{folder_path}/*.csv")
    df_list = []

    for file in file_paths:
        export_time = os.path.getmtime(file)
        export_time_datetime = convert_to_datetime(time.localtime(export_time))

        if file.endswith('.xlsx'):
            df = pl.read_excel(file, sheet_name=sheet_name)
        elif file.endswith('.csv'):
            try:
                df = pl.read_csv(file, encoding="utf-8", infer_schema_length=5000)
            except:
                df = pl.read_csv(file, encoding="ISO-8859-1", ignore_errors=True, infer_schema_length=5000)

        df = df.with_columns([
            pl.col(col).cast(pl.String)
            for col in df.columns
        ])

        df = df.with_columns([
            pl.lit(os.path.basename(file)).alias('File Name'),
            pl.lit(export_time_datetime).alias('Export Time')
        ])

        df_list.append(df)

    if df_list:
        merged_df = pl.concat(df_list, how='vertical')
    else:
        merged_df = pl.DataFrame()

    return merged_df

def input_data_parquet(folder_path):
    file_paths = glob.glob(f"{folder_path}/*.parquet")
    df_list = []

    for file in file_paths:
        export_time = os.path.getmtime(file)
        export_time_datetime = convert_to_datetime(time.localtime(export_time))

        df = pl.read_parquet(file)

        df = df.with_columns([
            pl.lit(os.path.basename(file)).alias('File Name'),
            pl.lit(export_time_datetime).alias('Export Time')
        ])
        df_list.append(df)

    if not df_list:
        return pl.DataFrame()

    all_columns = {col for df in df_list for col in df.columns}

    aligned_list = []
    for df in df_list:
        for col in all_columns - set(df.columns):
            df = df.with_columns(pl.lit(None).alias(col))

        new_schema = {} 
        for col in df.columns:
            if col in ["File Name", "Export Time"]:
                continue
            dtype = df[col].dtype
            if dtype.is_numeric():
                new_schema[col] = pl.Float64

        df = df.cast(new_schema)
        aligned_list.append(df)

    merged_df = pl.concat(aligned_list, how="vertical")
    return merged_df

def csv_to_parquet(input_folder, output_folder=None):
    import polars as pl
    import glob, os

    if output_folder is None:
        output_folder = input_folder
    os.makedirs(output_folder, exist_ok=True)
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    if not csv_files:
        return
    for csv_file in csv_files:
        try:
            df = pl.read_csv(
                csv_file,
                infer_schema_length=10000,
                schema_overrides={
                    "Promoter Score (Calc)": pl.Float64,
                    "Detractor Score (Calc)": pl.Float64,
                }
            )
            parquet_file = os.path.splitext(os.path.basename(csv_file))[0] + ".parquet"
            parquet_path = os.path.join(output_folder, parquet_file)
            df.write_parquet(parquet_path)
        except Exception as e:
            print(f"Không convert được file: {csv_file} - Lỗi: {e}")
        
today_temp = datetime.today().date()
today = today_temp.strftime('%b_%d_%Y')

In [15]:
first_glob_1 = "C:/Users/huuchinh.nguyen"
first_glob_2 = "C:/Users/ADMIN"

if os.path.exists(first_glob_1):
    first_glob = first_glob_1
elif os.path.exists(first_glob_2):
    first_glob = first_glob_2
else:
    raise FileNotFoundError(f"Neither {first_glob_1} nor {first_glob_2} exists.")

folder_paths = {
    "input_performance":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_NON_EN/new_look_excel_data',
    "output_non_en_performance_global":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Global Report/performance_non_en',
    "input_survey":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/SURVEY_EN',
    "input_t3":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/T3_EN',
    "input_oa":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/OA',
    "mapping_file": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Global Report/LOB Mapping Aligned OneReport/AQG_summarized.xlsx',
    "t3_invalid": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Global Report/T3 Validation Sheet-2025.xlsx',
    "global_hc": f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/Global HC.xlsx',
    "input_parquet_performance":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/RAW/NEW_LOOK_EXCEL_NON_EN/parquet_rawdata'
}

In [16]:
csv_to_parquet(folder_paths["input_performance"], folder_paths["input_parquet_performance"])

In [17]:
SURVEY_INPUT = input_data(folder_paths["input_survey"])

survey_added_ir = SURVEY_INPUT.with_columns([
    pl.when(
        (pl.col("Question Category") == "Resolution") & (pl.col("Answer") == "Yes")
    ).then(1)
    .when(
        (pl.col("Question Category") == "Resolution") & (pl.col("Answer") == "No")
    ).then(0)
    .otherwise(0)
    .alias("_ir")
])

survey_ae = (
    SURVEY_INPUT
    .filter(pl.col("Question Category") == "agentExperience")
    .with_columns(
        pl.col("Answer")
          .cast(pl.Int64, strict=False)
          .alias("_ae")
    )
)

survey_verbatim = (
    SURVEY_INPUT
    .filter(pl.col("Question Category") == "Feedback")
    .with_columns([
        pl.col("Answer").alias("_verbatim"),
        # Joined Date (only date part)
        pl.col("Joined Time")
          .str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False)
          .dt.date()
          .alias("Joined Date"),
        # Joined Month in YY_MM format (e.g., 25_01)
        pl.col("Joined Time")
          .str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False)
          .dt.strftime("%y_%m")
          .alias("Joined Month"),
    ])
)

metrics = ["d_happy_response", "d_surprised_response", "e_response", "t_response", "u_response"]

survey_duet = (
    SURVEY_INPUT
    .filter(pl.col("Question Category").is_in(metrics))
    .with_columns([
        pl.col("Question Category").alias("metric"),
        pl.col("Answer").cast(pl.Float64, strict=False).alias("value")
    ])
    .group_by(["Conversation Id", "metric"])
    .agg(pl.max("value").alias("value")) 
    .pivot(values="value", index="Conversation Id", columns="metric")
)

# --- Excel-equivalent calculations using column names (force Float64) ---
survey_duet = survey_duet.with_columns([
    pl.when(pl.any_horizontal([pl.col("d_happy_response").is_null(),
                               pl.col("d_surprised_response").is_null()]))
      .then(pl.lit(None, dtype=pl.Float64))
      .otherwise(pl.when((pl.col("d_happy_response") >= 4) & (pl.col("d_surprised_response") >= 3))
                   .then(100.0).otherwise(0.0))
      .alias("delight"),

    pl.when(pl.col("u_response").is_null()).then(pl.lit(None, dtype=pl.Float64))
      .otherwise(((pl.col("u_response") - 1.0) / 6.0) * 100.0).alias("usability"),

    pl.when(pl.col("e_response").is_null()).then(pl.lit(None, dtype=pl.Float64))
      .otherwise(((pl.col("e_response") - 1.0) / 6.0) * 100.0).alias("ease"),

    pl.when(pl.col("t_response").is_null()).then(pl.lit(None, dtype=pl.Float64))
      .otherwise(((pl.col("t_response") - 1.0) / 6.0) * 100.0).alias("trust"),

    pl.when(pl.any_horizontal([pl.col("d_happy_response").is_null(),
                               pl.col("d_surprised_response").is_null(),
                               pl.col("u_response").is_null(),
                               pl.col("e_response").is_null(),
                               pl.col("t_response").is_null()]))
      .then(pl.lit(None, dtype=pl.Float64))
      .otherwise(pl.when((pl.col("d_happy_response") >= 5) &
                         (pl.col("d_surprised_response") >= 3) &
                         ((pl.col("u_response") + pl.col("e_response")) >= 13) &
                         (pl.col("e_response") >= 6) &
                         (pl.col("t_response") >= 6))
                   .then(100.0).otherwise(0.0))
      .alias("DUET"),
])

# Round computed columns
survey_duet = survey_duet.with_columns([
    pl.col("delight").round(0),
    pl.col("usability").round(0),
    pl.col("ease").round(0),
    pl.col("trust").round(0),
    pl.col("DUET").round(0),
])

# Keep ALL required columns (raw + computed)
final_cols = ["Conversation Id"] + metrics + ["delight", "usability", "ease", "trust", "DUET"]
survey_duet = survey_duet.select([c for c in final_cols if c in survey_duet.columns])
survey_ir = (survey_added_ir.select(["Conversation Id", "_ir"]).filter(pl.col("_ir") == 1).unique())
survey_ae = (survey_ae.select(["Conversation Id", "_ae"]).filter(pl.col("_ae") > 0).unique())
verbatim = (survey_verbatim.select(["Conversation Id", "Agent Email ID", "Joined Month", "Joined Date", "Joined Time", "Offered Time", "Submitted Time", "_verbatim"])).unique()

print(survey_duet.columns)
survey_duet

C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_29844\1680620612.py:53: DeprecationWarning: The argument `columns` for `DataFrame.pivot` is deprecated. It has been renamed to `on`.
  .pivot(values="value", index="Conversation Id", columns="metric")


['Conversation Id', 'd_happy_response', 'd_surprised_response', 'e_response', 't_response', 'u_response', 'delight', 'usability', 'ease', 'trust', 'DUET']


Conversation Id,d_happy_response,d_surprised_response,e_response,t_response,u_response,delight,usability,ease,trust,DUET
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""6ff78b0b-7e16-4b57-b290-f18e1a…",4.0,1.0,7.0,7.0,7.0,0.0,100.0,100.0,100.0,0.0
"""39c2fcdd-b86f-41ea-8c70-60e3d7…",5.0,3.0,7.0,7.0,7.0,100.0,100.0,100.0,100.0,100.0
"""dd3f8fa0-5bcb-4b1c-9a61-efa3f3…",5.0,5.0,7.0,7.0,7.0,100.0,100.0,100.0,100.0,100.0
"""ef44cc6f-b4d6-4f5a-8459-330a85…",1.0,1.0,7.0,7.0,7.0,0.0,100.0,100.0,100.0,0.0
"""e7236446-bfa1-4549-bc6b-30c0b6…",5.0,5.0,7.0,7.0,7.0,100.0,100.0,100.0,100.0,100.0
…,…,…,…,…,…,…,…,…,…,…
"""a9c2bad8-25b8-4a32-873e-76e645…",4.0,4.0,7.0,7.0,5.0,100.0,67.0,100.0,100.0,0.0
"""979a3d09-75b1-41e7-b772-9e1ed7…",2.0,5.0,4.0,4.0,4.0,0.0,50.0,50.0,50.0,0.0
"""c265b229-9348-4bcd-9508-ba3a69…",5.0,5.0,7.0,7.0,7.0,100.0,100.0,100.0,100.0,100.0


In [18]:
T3_INPUT = input_data(folder_paths["input_t3"])

t3_added_col_t3 = T3_INPUT.with_columns([
    pl.when(
        (pl.col("Agent Queue Group Name").str.contains("T3", literal=True)) &
        (pl.col("Transferred (Yes/No)") == "Yes")
    ).then(1).otherwise(0).alias("T3")])

t3_final = (t3_added_col_t3.select(["Conversation Id", "T3"]).filter(pl.col("T3") == 1).unique())
t3_final

Conversation Id,T3
str,i32
"""4e14a575-a63a-4c6d-bf82-353b88…",1
"""7089696d-a9b1-4d92-8a06-d179a7…",1
"""12256931-7f34-4786-b490-893fa5…",1
"""f009bca6-daaa-4de0-98c0-629bf7…",1
"""d4621c4d-ea9d-425e-8ae7-a690e0…",1
…,…
"""d0f91275-1eeb-44e8-ad54-19e92c…",1
"""385e4e04-99ba-4eb1-97b2-599cbe…",1
"""149df64f-8859-45b2-a35a-6596d7…",1


In [19]:
T3_INVALID_INPUT = pl.read_excel(folder_paths["t3_invalid"], sheet_name="Invalid Cases").select(["Conversation Id", "Case Eligible to be Disputed"])
T3_INVALID_INPUT = T3_INVALID_INPUT.cast({"Conversation Id": pl.String,"Case Eligible to be Disputed": pl.String})

t3_invalid = T3_INVALID_INPUT.with_columns([pl.when(pl.col("Case Eligible to be Disputed") == "Accepted").then(1).otherwise(0).alias("Invalid T3")])

t3_invalid_final = (t3_invalid.filter(pl.col("Invalid T3") == 1).select(["Conversation Id", "Invalid T3"]).unique())

t3_invalid_final

Conversation Id,Invalid T3
str,i32
"""edcb6a88-d3a3-47fd-88ed-aa3cde…",1
"""d64b7928-1a47-41f3-a13b-c40a59…",1
"""fb4cee7e-0f76-447e-879e-c4e604…",1
"""64a5ae33-c4c3-429d-af8e-1f396d…",1
"""3bebd67b-a525-4ea4-84c0-d3e330…",1
…,…
"""9aa4e723-484c-4855-aa95-cdf436…",1
"""de0ea329-6361-441c-9bd9-cfd73e…",1
"""8bb9bbf8-9dad-4f81-bdec-12f44d…",1


In [20]:
OA_INPUT = input_data(folder_paths["input_oa"])

OA_INPUT = OA_INPUT.with_columns([
    pl.col("Handle (Count)").cast(pl.Int64, strict=False),
    pl.col("Outbound (Count)").cast(pl.Int64, strict=False)
])

grouped_OA = (
    OA_INPUT.group_by('Conversation Id')
    .agg([
        pl.col('Handle (Count)').sum().alias('HC'),
        pl.col('Outbound (Count)').sum().alias('OA')
    ])
)

oa_final = grouped_OA.select(['Conversation Id', 'OA', 'HC']).unique()
oa_final

Conversation Id,OA,HC
str,i64,i64
"""125f2921-5c6e-46bc-9dd7-585722…",2,1
"""80d40428-7842-4158-97e2-5f2eeb…",2,1
"""e65249bb-958a-4fc7-a410-1ba5ef…",0,1
"""fc42d9c0-9a2f-4e07-809b-df2985…",0,1
"""407d058d-b504-47e9-8877-bc559b…",0,1
…,…,…
"""71c8148e-59fc-4fa7-b458-67a3ed…",4,1
"""be474548-d5d1-4475-8ed6-32072e…",1,1
"""3967f856-7b48-4882-af8d-dd772b…",0,1


In [21]:
PERFORMANCE_INPUT = input_data_parquet(folder_paths["input_parquet_performance"])

try: 
    if PERFORMANCE_INPUT.columns[0] == "":
        PERFORMANCE_INPUT = PERFORMANCE_INPUT.drop(PERFORMANCE_INPUT.columns[0])
except: pass

existing_cols = set(PERFORMANCE_INPUT.columns)

columns_to_cast = {
    "Joined Time": pl.Datetime,
    "Handle Time (Sum)": pl.Float64,
    "Handle (Count)": pl.Int64,
    "Hold Time (Sum)": pl.Float64,
    "Talk Time (Sum)": pl.Float64,
    "Wrap Up Time (Sum)": pl.Float64,
    "Survey Submitted (Count)":pl.Float64,
    "Promoter Score (Calc)": pl.Float64,
    "Detractor Score (Calc)": pl.Float64,
    "Neutral Score (Calc)": pl.Float64,
    "Has Followup Time": pl.Float64,
    "Response Time (Sum)": pl.Float64,
    "Response Time (Avg)": pl.Float64,
    "Concurrency": pl.Float64,
    "Survey Offered (Count)":pl.Int64,
    "Sum of Chat Agent First Response Time":pl.Float64
}

datetime_cols = ["Started Time", "Joined Time", "Submitted Time", "Left Time"]

casts = []

for col, dtype in columns_to_cast.items():
    if col in existing_cols:
        if col == "Submitted Date":
            casts.append(pl.col(col).str.strptime(pl.Date, "%m/%d/%Y", strict=False))
        elif col in datetime_cols:
            casts.append(
                pl.when(pl.col(col).str.contains(r"^\d{4}-\d{2}-\d{2}"))  # ISO: YYYY-MM-DD
                  .then(pl.col(col).str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False))
                .when(pl.col(col).str.contains(r"^\d{1,2}/\d{1,2}/\d{4}"))  # US: M/D/YYYY
                  .then(pl.col(col).str.strptime(pl.Datetime, "%m/%d/%Y %H:%M", strict=False))
                .otherwise(None)
                .alias(col)
            )
        elif dtype == pl.Float64:
            casts.append(
                pl.col(col)
                .cast(pl.Utf8)
                .str.replace_all(",", "")
                .cast(pl.Float64, strict=False)
                .alias(col)
            )
        else:
            casts.append(pl.col(col).cast(dtype).alias(col))

PERFORMANCE_CHANGED_TYPE = PERFORMANCE_INPUT.with_columns(casts)
PERFORMANCE_ADDED_CCR72 = PERFORMANCE_CHANGED_TYPE.with_columns([
    pl.when(pl.col("Has Followup Time").cast(pl.Float64) <= 4320)
      .then(1.0)
      .otherwise(0.0)
      .alias("CCR72")
])

PERFORMANCE_NEXT_STEP = PERFORMANCE_ADDED_CCR72.with_columns([
    pl.col("Joined Time").dt.date().alias("Joined Date")
])

GLOBAL_HC = pl.read_excel(folder_paths["global_hc"])
merged_global_hc = PERFORMANCE_NEXT_STEP.join(GLOBAL_HC[['SSO ID', "Name", "Supervisor", "Production Start date", "Agent/Non Agent"]], left_on=["Agent Email ID"], right_on=['SSO ID'], how='left')


mapping = pl.read_excel(folder_paths["mapping_file"], sheet_name="queue_name_mapping")

performance_cleaned = merged_global_hc.join(mapping, on="Agent Queue Group Name",how='left')
LOB_MAPPING_SHOWED = mapping.filter(
    pl.col("Agent Queue Group Name").str.contains(r"[dD][aA]")
)

print(LOB_MAPPING_SHOWED)

shape: (12, 5)
┌────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬───────────────┐
│ MainLOBs       ┆ CFG Groups          ┆ LOB                 ┆ Agent Queue Group   ┆ Site          │
│ ---            ┆ ---                 ┆ ---                 ┆ Name                ┆ ---           │
│ str            ┆ str                 ┆ str                 ┆ ---                 ┆ str           │
│                ┆                     ┆                     ┆ str                 ┆               │
╞════════════════╪═════════════════════╪═════════════════════╪═════════════════════╪═══════════════╡
│ Retail Non     ┆ Voice Danish        ┆ Voice Danish        ┆ Voice_GLB_DA_Expert ┆ Concentrix    │
│ English        ┆                     ┆                     ┆                     ┆ (Helsingborg) │
│ Retail Non     ┆ Voice Danish        ┆ Voice Danish        ┆ Voice_GLB_DA_Lodgin ┆ Concentrix    │
│ English        ┆                     ┆                     ┆ g            

In [22]:
print(performance_cleaned.columns)

['Joined Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Queue Group Name', 'Latest VA Intent', 'Agent Business Location', 'Initiated Outbound (Yes / No)', 'Latest VA Product', 'Response Count', 'Response Time', 'Customer Loyalty Status', 'Business Segment Name', 'Partner Name', 'Language', 'Effective Disconnected By', 'Joined Time', 'Left Time', 'Entry Point App', 'Device Category', 'Device Os', 'Browser', 'Has Followup Time', 'Queue Name', 'Agent Manager Name', 'Agent Name', 'Handle (Count)', 'Handle Time (Sum)', 'Wrap Up Time (Sum)', 'Hold Time (Sum)', 'Talk Time (Sum)', 'Follow Up (Count)', 'Survey Submitted (Count)', 'Promoter Score (Calc)', 'Detractor Score (Calc)', 'Assigned Agent Time (Sum)', 'Sum of Chat Agent First Response Time', 'Survey Offered (Count)', 'File Name', 'Export Time', 'CCR72', 'Name', 'Supervisor', 'Production Start date', 'Agent/Non Agent', 'MainLOBs', 'CFG Groups', 'LOB', 'Site']


In [23]:
lc_scheme = {
    "LOB": ["NL Chat", "LG Chat","NL Chat", "LG Chat"],
    "Threshole_LC": [2700, 1500,
           2400, 1200],
    "Effective Date": [date(2023, 12, 1), date(2023, 12, 1), 
                       date(2025, 7, 1), date(2025, 7, 1)]
}
lc_scheme = pl.DataFrame(lc_scheme)
print(lc_scheme)

shape: (4, 3)
┌─────────┬──────────────┬────────────────┐
│ LOB     ┆ Threshole_LC ┆ Effective Date │
│ ---     ┆ ---          ┆ ---            │
│ str     ┆ i64          ┆ date           │
╞═════════╪══════════════╪════════════════╡
│ NL Chat ┆ 2700         ┆ 2023-12-01     │
│ LG Chat ┆ 1500         ┆ 2023-12-01     │
│ NL Chat ┆ 2400         ┆ 2025-07-01     │
│ LG Chat ┆ 1200         ┆ 2025-07-01     │
└─────────┴──────────────┴────────────────┘


In [24]:
performance_processed = performance_cleaned.with_columns([
    (1 - pl.col("Promoter Score (Calc)") - pl.col("Detractor Score (Calc)"))
        .alias("Neutral Score (Calc)")
])

performance_processed = performance_processed.with_columns([
    (pl.col("Promoter Score (Calc)") * pl.col("Survey Submitted (Count)")).round(0).alias("_promoter"),
    (pl.col("Detractor Score (Calc)") * pl.col("Survey Submitted (Count)")).round(0).alias("_detractor"),
    (pl.col("Neutral Score (Calc)") * pl.col("Survey Submitted (Count)")).round(0).alias("_neutral")
])

performance_processed = performance_processed.with_columns([

    pl.when(pl.col("Initiated Outbound (Yes / No)") == "Yes").then(1).otherwise(0).alias("_aob"),

    pl.col("Survey Submitted (Count)").alias("_survey"),
    pl.col("CCR72").cast(pl.Float64).fill_null(0.0).alias("_fup_72"),
    (1.0 - pl.col("CCR72").cast(pl.Float64).fill_null(0.0)).alias("_rr"),
    pl.when(pl.col("Hold Time (Sum)") > 300).then(1).otherwise(0).cast(pl.Int64).alias("LH"),
    pl.when(
        ((pl.col("Latest VA Product") == "FLIGHT") & (pl.col("Handle Time (Sum)") > 2700))
        | 
        ((pl.col("Latest VA Product") != "FLIGHT") & (pl.col("Handle Time (Sum)") > 1500))
    ).then(1).otherwise(0).cast(pl.Int64).alias("_lc"),
    pl.col("Joined Time").dt.date().alias("_PST.Date"),
    pl.col("Joined Time").dt.strftime("%y_%m").alias("_PST.Month"),
    pl.col("Joined Time").dt.strftime("%y_W%V").alias("_PST.WeekNo"),
    (pl.col("Joined Time") - pl.col("Joined Time").dt.weekday() * pl.duration(days=1)).dt.strftime("WB_%y/%m/%d").alias("_PST.Week"),
    pl.col("Joined Time").dt.year().alias("_PST.Year"),
    pl.concat_str([
        pl.col("Agent Email ID").cast(pl.Utf8).fill_null(""),
        pl.col("Conversation Id").cast(pl.Utf8).fill_null(""),
        pl.col("Joined Time").dt.strftime("%y%m%d%H%M%S")
    ], separator="_").alias("_conver_unique"),
    pl.concat_str([
        pl.col("Agent Email ID").cast(pl.Utf8).fill_null(""),
        pl.col("Conversation Id").cast(pl.Utf8).fill_null("")], 
        separator="_").alias("_key"),
    pl.when(pl.col("Survey Offered (Count)") > 0).then(1).otherwise(0).alias("_offer"),
    pl.when(pl.col("Joined Time") <= pl.datetime(2025, 7, 7)).then(pl.lit("Pre")).otherwise(pl.lit("Post")).alias("Pre/Post")
])

performance_processed = performance_processed.with_columns([
    (
        (pl.col("_PST.Date").cast(pl.Date) - pl.col("Production Start date").cast(pl.Date))
        / pl.duration(days=1)
    ).cast(pl.Int32).alias("AON_Days")
])
performance_processed = performance_processed.with_columns([
    pl.when(
        pl.col("AON_Days").is_null() & (
            (pl.col("Agent/Non Agent") == "Agent") | (pl.col("Agent/Non Agent") == "ID Deleted")
        )
    )
    .then(pl.lit("Nesting"))
    .when(pl.col("AON_Days") > 180)
      .then(pl.lit("> 180 Days"))
    .when(pl.col("AON_Days") >= 91)
      .then(pl.lit("91 - 180"))
    .when(pl.col("AON_Days") >= 61)
      .then(pl.lit("61 - 90"))
    .when(pl.col("AON_Days") >= 31)
      .then(pl.lit("31 - 60"))
    .when(pl.col("AON_Days") >= 0)
      .then(pl.lit("00 - 30"))
    .otherwise(None)
    .alias("AON Status")
])
# performance_processed = performance_processed.join_asof(
#     lc_scheme,
#     left_on="_PST.Date",
#     right_on="Effective Date",
#     by="LOB",
#     strategy="backward"
# )
# performance_processed = performance_processed.with_columns([
#     (pl.col("Handle Time (Sum)") >= pl.col("Threshole_LC")).cast(pl.Int8).alias("_lc")
# ])

performance_processed = performance_processed.with_columns([
    pl.concat_str([pl.col("Agent Email ID"), pl.col("_PST.Date").dt.strftime("%y%m%d")]).alias("KEY")
])

performance_processed = performance_processed.with_columns([
    (
        pl.when(pl.col("_promoter") > 0)
          .then(pl.lit("promoter, "))
          .otherwise(pl.lit(""))
      +
        pl.when(pl.col("_detractor") > 0)
          .then(pl.lit("detractor, "))
          .otherwise(pl.lit(""))
      +
        pl.when(pl.col("_neutral") > 0)
          .then(pl.lit("neutral"))
          .otherwise(pl.lit(""))
    ).str.strip_chars(", ").alias("_nps_type")
])

# def get_30min_interval(dt):
#     if dt is None:
#         return None
#     hour = dt.hour
#     minute = dt.minute
#     if minute < 30:
#         start = datetime(dt.year, dt.month, dt.day, hour, 0)
#         end = start + timedelta(minutes=29)
#     else:
#         start = datetime(dt.year, dt.month, dt.day, hour, 30)
#         end = start + timedelta(minutes=29)
#     return f"{start.strftime('%H:%M')}-{end.strftime('%H:%M')}"

# def get_period(dt):
#     if dt is None:
#         return None
#     hour = dt.hour
#     if hour >= 18:
#         return 'Night'
#     elif hour >= 12:
#         return 'Mid'
#     else:
#         return 'Morning'

# performance_processed = performance_processed.with_columns([
#     pl.col('Joined Time').map_elements(get_30min_interval, return_dtype=pl.Utf8).alias('_PST.Interval'),
#     pl.col('Join Time (VNT)').map_elements(get_30min_interval, return_dtype=pl.Utf8).alias('_VNT.Interval'),
#     pl.col('Datetime_First_Start_Shift').map_elements(get_period, return_dtype=pl.Utf8).alias('_VNT.Period')
# ])

performance_merged_survey_t3 = (
    performance_processed
    .join(survey_ir, left_on = "Conversation Id", right_on = "Conversation Id", how="left")
    .join(survey_ae, left_on = "Conversation Id", right_on = "Conversation Id", how="left")
    .join(survey_duet, left_on = "Conversation Id", right_on = "Conversation Id", how="left")
    .join(t3_final, left_on = "Conversation Id", right_on = "Conversation Id", how="left")
    .join(t3_invalid_final, left_on = "Conversation Id", right_on = "Conversation Id", how="left")
    .join(oa_final, left_on = "Conversation Id", right_on = "Conversation Id", how="left")
)

print(performance_merged_survey_t3.columns)
print(performance_merged_survey_t3.select(["_PST.Date", "Production Start date", "AON_Days"]).head())

selected_columns = [
    "_PST.Date","Agent People Id","Conversation Id","Agent Email ID","Joined Time","Left Time","Agent Queue Group Name","Latest VA Intent","Latest VA Product","Business Segment Name",
    "Partner Name","Handle (Count)","Handle Time (Sum)","Response Count","Response Time","Hold Time (Sum)","Wrap Up Time (Sum)",
    "Talk Time (Sum)","LOB","_aob","CCR72","_offer","_survey","_promoter","_detractor","_neutral","_nps_type","Name","Supervisor","Agent Business Location",
    "AON Status","Customer Loyalty Status","Language","Effective Disconnected By","Sum of Chat Agent First Response Time","_PST.Year",
    "_PST.WeekNo","_PST.Week","T3","_ir","Agent/Non Agent","_ae","_PST.Month","OA","HC","LH","_lc","Invalid T3","_conver_unique","d_happy_response","d_surprised_response",
    "e_response","t_response","u_response",'delight', 'usability', 'ease', 'trust', 'DUET'
]

missing_cols = [col for col in selected_columns if col not in performance_merged_survey_t3.columns]
print("Các cột bị thiếu:", missing_cols)

performance_filtered = performance_merged_survey_t3.select(selected_columns)
performance_filtered = performance_filtered.unique()

['Joined Date', 'Agent People Id', 'Conversation Id', 'Agent Email ID', 'Agent Queue Group Name', 'Latest VA Intent', 'Agent Business Location', 'Initiated Outbound (Yes / No)', 'Latest VA Product', 'Response Count', 'Response Time', 'Customer Loyalty Status', 'Business Segment Name', 'Partner Name', 'Language', 'Effective Disconnected By', 'Joined Time', 'Left Time', 'Entry Point App', 'Device Category', 'Device Os', 'Browser', 'Has Followup Time', 'Queue Name', 'Agent Manager Name', 'Agent Name', 'Handle (Count)', 'Handle Time (Sum)', 'Wrap Up Time (Sum)', 'Hold Time (Sum)', 'Talk Time (Sum)', 'Follow Up (Count)', 'Survey Submitted (Count)', 'Promoter Score (Calc)', 'Detractor Score (Calc)', 'Assigned Agent Time (Sum)', 'Sum of Chat Agent First Response Time', 'Survey Offered (Count)', 'File Name', 'Export Time', 'CCR72', 'Name', 'Supervisor', 'Production Start date', 'Agent/Non Agent', 'MainLOBs', 'CFG Groups', 'LOB', 'Site', 'Neutral Score (Calc)', '_promoter', '_detractor', '_neut

In [25]:
# 1) Read the removal list from Excel
removed_id = pl.read_excel(folder_paths["mapping_file"], sheet_name="id_removed")

# 2) Helper to normalize Conversation Id (cast -> drop quotes -> trim spaces)
def clean_id(expr: pl.Expr) -> pl.Expr:
    return (
        expr.cast(pl.Utf8)
            .str.replace_all(r'["“”]', "")      # remove any quote characters
            .str.replace_all(r'^\s+|\s+$', "")  # trim leading/trailing whitespace (regex)
    )

# Validate required column
if "Conversation Id" not in removed_id.columns:
    raise ValueError("Sheet 'id_removed' must contain a 'Conversation Id' column.")

# Canonical removal list: unique, non-null, non-empty
removed_id_norm = (
    removed_id
    .select(clean_id(pl.col("Conversation Id")).alias("Conversation Id"))
    .drop_nulls()
    .filter(pl.col("Conversation Id") != "")
    .unique()
)

# 3) Filter performance by anti-join (keep rows NOT present in removed list)
before = performance_filtered.height

performance_filtered = (
    performance_filtered
    .with_columns(clean_id(pl.col("Conversation Id")).alias("Conversation Id"))
    .join(removed_id_norm, on="Conversation Id", how="anti")
)

after = performance_filtered.height
print(f"Removed {before - after} rows by Conversation Id; remaining: {after}")

Removed 51 rows by Conversation Id; remaining: 565853


In [26]:
performance_unique = performance_filtered.unique()

performance_all_site = performance_unique

In [27]:
for month, group in performance_all_site.group_by('_PST.Month'):
    month_value = month[0]
    file_name = f"{month_value}.csv"
    file_path = os.path.join(folder_paths["output_non_en_performance_global"], file_name)
    
    group.write_csv(file_path)

for month, group in performance_all_site.group_by('_PST.Month', maintain_order=True):
    month_value = month[0]
    file_name = f"{month_value}_performance_en_global.parquet"
    file_path = os.path.join(folder_paths["output_non_en_performance_global"], file_name)
    group.write_parquet(file_path)

file_path_all = os.path.join(folder_paths["output_non_en_performance_global"], "all_months_performance_non_en_global.parquet")
performance_all_site.write_parquet(file_path_all)

In [28]:
print(performance_all_site['_PST.Month'].unique())

shape: (10,)
Series: '_PST.Month' [str]
[
	"25_04"
	"25_05"
	"25_02"
	"25_03"
	"25_09"
	"25_08"
	"25_07"
	"25_06"
	"25_10"
	"25_01"
]


In [29]:
print(performance_all_site.schema)

Schema({'_PST.Date': Date, 'Agent People Id': Float64, 'Conversation Id': String, 'Agent Email ID': String, 'Joined Time': Datetime(time_unit='us', time_zone=None), 'Left Time': String, 'Agent Queue Group Name': String, 'Latest VA Intent': String, 'Latest VA Product': String, 'Business Segment Name': String, 'Partner Name': String, 'Handle (Count)': Int64, 'Handle Time (Sum)': Float64, 'Response Count': Float64, 'Response Time': Float64, 'Hold Time (Sum)': Float64, 'Wrap Up Time (Sum)': Float64, 'Talk Time (Sum)': Float64, 'LOB': String, '_aob': Int32, 'CCR72': Float64, '_offer': Int32, '_survey': Float64, '_promoter': Float64, '_detractor': Float64, '_neutral': Float64, '_nps_type': String, 'Name': String, 'Supervisor': String, 'Agent Business Location': String, 'AON Status': String, 'Customer Loyalty Status': String, 'Language': String, 'Effective Disconnected By': String, 'Sum of Chat Agent First Response Time': Float64, '_PST.Year': Int32, '_PST.WeekNo': String, '_PST.Week': String

In [30]:
x = performance_all_site.filter(pl.col("_PST.Month") == "25_07")

nps_by_lob_location = (
    x.group_by(["LOB", "Agent Business Location"])
     .agg([
         pl.col("_survey").sum().alias("Survey"),
         pl.col("_promoter").sum().alias("Promoter"),
         pl.col("_detractor").sum().alias("Detractor"),
         pl.col("_neutral").sum().alias("Neutral"),
         pl.col("_conver_unique").n_unique().alias("Volume"),
         pl.when(pl.col("_lc") == 1)
           .then(pl.col("_conver_unique"))
           .otherwise(None)
           .n_unique()
           .alias("LC_Count"),

         pl.col("Wrap Up Time (Sum)").sum().alias("Wrap Up Time (Sum)"),
         pl.col("Handle (Count)").sum().alias("Handle (Count)"),
         pl.col("Hold Time (Sum)").sum().alias("Hold Time (Sum)"),
         pl.col("Handle Time (Sum)").sum().alias("Handle Time (Sum)"),
         pl.col("_aob").sum().alias("Outbound"),
         pl.col("_survey").sum().alias("Survey (Sum)"),
         pl.col("T3").sum().alias("T3 (Sum)"),
         pl.col("_offer").sum().alias("Offer (Sum)")
     ])
     .with_columns([
         ((pl.col("Promoter") - pl.col("Detractor")) / pl.col("Survey") * 100).alias("NPS"),
         ((pl.col("Promoter") + pl.col("Neutral") - pl.col("Detractor")) / pl.col("Survey") * 100).alias("NPS_7"),
         ((pl.col("LC_Count") / pl.col("Volume")) * 100).alias("LC"),
         (pl.col("Wrap Up Time (Sum)") / pl.col("Handle (Count)")).alias("ACW"),
         ((pl.col("Outbound") / pl.col("Handle (Count)"))*100).alias("AOB"),
         ((pl.col("Hold Time (Sum)") / pl.col("Handle Time (Sum)"))*100).alias("Hold"),
         ((pl.col("Survey (Sum)") / pl.col("Handle (Count)"))*100).alias("SRR"),
         ((pl.col("T3 (Sum)") / pl.col("Handle (Count)"))*100).alias("T3%"),
     ])
)

aht_df = (
    x.group_by(["LOB", "Agent Business Location", "_conver_unique"])
     .agg(pl.col("Handle Time (Sum)").max().alias("_ht"))
     .group_by(["LOB", "Agent Business Location"])
     .agg(pl.col("_ht").mean().alias("AHT"))
)

nps_by_lob_location = nps_by_lob_location.join(aht_df, on=["LOB", "Agent Business Location"], how="left")

filtered_sorted_nps = (
    nps_by_lob_location
    .with_columns([
        pl.col("NPS").cast(pl.Float64, strict=False).round(1),
        pl.col("AHT").cast(pl.Float64, strict=False).round(1),
        pl.col("LC").cast(pl.Float64, strict=False).round(1),
        pl.col("ACW").cast(pl.Float64, strict=False).round(1),
        pl.col("AOB").cast(pl.Float64, strict=False).round(1),
        pl.col("Hold").cast(pl.Float64, strict=False).round(1),
        pl.col("SRR").cast(pl.Float64, strict=False).round(1),
        pl.col("NPS_7").cast(pl.Float64, strict=False).round(1),
        pl.col("T3%").cast(pl.Float64, strict=False).round(1),
        pl.col("Offer (Sum)").cast(pl.Float64, strict=False).round(1)
    ])
    .select([
        "LOB", "Agent Business Location", "NPS", "AHT", "LC", "ACW", "AOB", "Hold", 
        "Volume", "Survey", "Promoter", "Detractor", "Neutral", "NPS_7", "T3%", "SRR", "Offer (Sum)"
    ])
    .sort(
        by=["LOB", "Agent Business Location", "NPS", "AHT", "LC", "ACW", "AOB", "Hold", "Volume", "Survey", "Promoter", "Detractor", "Neutral", "NPS_7", "T3%", "SRR", "Offer (Sum)"],
        descending=[False, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
    )
)
filtered_sorted_nps

LOB,Agent Business Location,NPS,AHT,LC,ACW,AOB,Hold,Volume,Survey,Promoter,Detractor,Neutral,NPS_7,T3%,SRR,Offer (Sum)
str,str,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64
"""Blended Dutch""","""Concentrix (Amsterdam)""",25.0,624.8,7.7,18.1,40.7,30.2,130,4.0,1.0,0.0,3.0,100.0,0.8,3.4,52.0
"""Blended Dutch""","""Concentrix (Paramaribo)""",-1.6,817.7,9.3,19.8,28.2,39.5,1992,61.0,17.0,18.0,26.0,41.0,0.1,3.1,595.0
"""Chat German""","""Concentrix (Cairo)""",64.4,745.5,5.2,0.0,0.1,0.0,3032,526.0,400.0,61.0,65.0,76.8,1.4,17.4,2903.0
"""Voice Danish""","""Concentrix (Helsingborg)""",12.8,913.4,12.7,21.1,24.1,28.0,2133,47.0,21.0,15.0,11.0,36.2,1.0,2.2,407.0
"""Voice EPS DE""","""Concentrix (Cairo)""",33.1,650.0,7.0,74.4,46.5,39.8,4486,133.0,76.0,32.0,25.0,51.9,0.9,3.3,717.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Voice German""","""Concentrix (Cairo)""",-5.3,756.8,11.1,55.2,44.9,43.1,696,19.0,8.0,9.0,2.0,5.3,2.0,3.4,127.0
"""Voice Northern""","""Concentrix (Helsingborg)""",0.0,693.0,6.4,28.3,26.0,30.7,4205,84.0,33.0,33.0,18.0,21.4,0.9,2.0,790.0
"""Voice Northern""","""Concentrix (Malaga)""",0.0,869.1,12.8,17.4,33.2,17.8,635,16.0,6.0,6.0,4.0,25.0,0.2,2.5,106.0


In [31]:
from datetime import date

x = performance_all_site.filter(pl.col("_PST.Date") == date(2025, 7, 16))


nps_by_lob_location = (
    x.group_by(["LOB", "Agent Business Location"])
     .agg([
         pl.col("_survey").sum().alias("Survey"),
         pl.col("_promoter").sum().alias("Promoter"),
         pl.col("_detractor").sum().alias("Detractor"),
         pl.col("_neutral").sum().alias("Neutral"),
         pl.col("_conver_unique").n_unique().alias("Volume"),
         pl.when(pl.col("_lc") == 1)
           .then(pl.col("_conver_unique"))
           .otherwise(None)
           .n_unique()
           .alias("LC_Count"),

         pl.col("Wrap Up Time (Sum)").sum().alias("Wrap Up Time (Sum)"),
         pl.col("Handle (Count)").sum().alias("Handle (Count)"),
         pl.col("Hold Time (Sum)").sum().alias("Hold Time (Sum)"),
         pl.col("Handle Time (Sum)").sum().alias("Handle Time (Sum)"),
         pl.col("_aob").sum().alias("Outbound"),
         pl.col("_survey").sum().alias("Survey (Sum)"),
         pl.col("T3").sum().alias("T3 (Sum)"),
         pl.col("_offer").sum().alias("Offer (Sum)")
     ])
     .with_columns([
         ((pl.col("Promoter") - pl.col("Detractor")) / pl.col("Survey") * 100).alias("NPS"),
         ((pl.col("Promoter") + pl.col("Neutral") - pl.col("Detractor")) / pl.col("Survey") * 100).alias("NPS_7"),
         ((pl.col("LC_Count") / pl.col("Volume")) * 100).alias("LC"),
         (pl.col("Wrap Up Time (Sum)") / pl.col("Handle (Count)")).alias("ACW"),
         ((pl.col("Outbound") / pl.col("Handle (Count)"))*100).alias("AOB"),
         ((pl.col("Hold Time (Sum)") / pl.col("Handle Time (Sum)"))*100).alias("Hold"),
         ((pl.col("Survey (Sum)") / pl.col("Handle (Count)"))*100).alias("SRR"),
         ((pl.col("T3 (Sum)") / pl.col("Handle (Count)"))*100).alias("T3%"),
     ])
)

aht_df = (
    x.group_by(["LOB", "Agent Business Location", "_conver_unique"])
     .agg(pl.col("Handle Time (Sum)").max().alias("_ht"))
     .group_by(["LOB", "Agent Business Location"])
     .agg(pl.col("_ht").mean().alias("AHT"))
)

nps_by_lob_location = nps_by_lob_location.join(aht_df, on=["LOB", "Agent Business Location"], how="left")

filtered_sorted_nps = (
    nps_by_lob_location
    .with_columns([
        pl.col("NPS").cast(pl.Float64, strict=False).round(1),
        pl.col("AHT").cast(pl.Float64, strict=False).round(1),
        pl.col("LC").cast(pl.Float64, strict=False).round(1),
        pl.col("ACW").cast(pl.Float64, strict=False).round(1),
        pl.col("AOB").cast(pl.Float64, strict=False).round(1),
        pl.col("Hold").cast(pl.Float64, strict=False).round(1),
        pl.col("SRR").cast(pl.Float64, strict=False).round(1),
        pl.col("NPS_7").cast(pl.Float64, strict=False).round(1),
        pl.col("T3%").cast(pl.Float64, strict=False).round(1),
        pl.col("Offer (Sum)").cast(pl.Float64, strict=False).round(1)
    ])
    .select([
        "LOB", "Agent Business Location", "NPS", "AHT", "LC", "ACW", "AOB", "Hold", 
        "Volume", "Survey", "Promoter", "Detractor", "Neutral", "NPS_7", "T3%", "SRR", "Offer (Sum)"
    ])
    .sort(
        by=["LOB", "Agent Business Location", "NPS", "AHT", "LC", "ACW", "AOB", "Hold", "Volume", "Survey", "Promoter", "Detractor", "Neutral", "NPS_7", "T3%", "SRR", "Offer (Sum)"],
        descending=[False, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
    )
)
filtered_sorted_nps

LOB,Agent Business Location,NPS,AHT,LC,ACW,AOB,Hold,Volume,Survey,Promoter,Detractor,Neutral,NPS_7,T3%,SRR,Offer (Sum)
str,str,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64
"""Blended Dutch""","""Concentrix (Amsterdam)""",0.0,962.7,66.7,13.3,66.7,43.5,3,1.0,0.0,0.0,1.0,100.0,0.0,33.3,2.0
"""Blended Dutch""","""Concentrix (Paramaribo)""",-33.3,667.6,4.3,18.7,18.1,39.1,94,3.0,0.0,1.0,2.0,33.3,0.0,3.2,18.0
"""Chat German""","""Concentrix (Cairo)""",50.0,762.8,9.8,0.0,0.0,0.0,61,14.0,10.0,3.0,1.0,57.1,3.3,23.0,57.0
"""Voice Danish""","""Concentrix (Helsingborg)""",NaN,852.4,11.6,21.9,24.7,31.8,95,0.0,0.0,0.0,0.0,NaN,1.1,0.0,9.0
"""Voice EPS DE""","""Concentrix (Cairo)""",50.0,577.4,6.8,67.5,47.5,37.7,147,4.0,2.0,0.0,2.0,100.0,0.0,3.3,20.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Voice German""","""Concentrix (Cairo)""",NaN,0.0,25.0,inf,NaN,NaN,4,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0
"""Voice Northern""","""Concentrix (Helsingborg)""",NaN,739.5,5.8,28.3,28.8,39.1,139,0.0,0.0,0.0,0.0,NaN,0.7,0.0,16.0
"""Voice Northern""","""Concentrix (Malaga)""",NaN,862.2,18.4,20.8,39.5,31.1,38,0.0,0.0,0.0,0.0,NaN,0.0,0.0,4.0
